# Suite No. 2 Upgrade: Izhikevich to V1-V4 Spectrolaminar Proxy Readouts

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/HNXJ/jaxfne/blob/main/tutorials/jaxfne_suite_no_2_spectrolaminar_motif.ipynb)

Pipeline: `Emitter -> Source -> Field -> Probe -> Objective -> Optimizer`.

## 1. Learning objectives

1. Configure one reduced Izhikevich emitter through `jaxfne`.
2. Compare E/PV/SST/VIP reduced-emitter presets through package-level arrays.
3. Build `net1`, a 100-emitter uniformly sampled 3D column.
4. Generate raster, LFP-like, CSD-like, EEG-proxy, MEG-proxy, and EMM-proxy figures without tutorial-local plotting functions.
5. Configure a V1-V4 six-layer scaffold with feedforward and feedback metadata.
6. Tune noise amplitude toward a firing-rate target with the Suite No. 2 AGSDR-Adam utility.

## 2. Biological/computational question

How far can a compact `jaxfne` grammar carry the workflow from one reduced emitter to a two-area laminar scaffold while preserving reusable source/field/probe readouts?

## 3. Mathematical glossary flow

### Reduced Izhikevich emitter

Formal equation:

$$rac{dv_i}{dt}=0.04v_i^2+5v_i+140-u_i+I_i^{native}(t), \qquad rac{du_i}{dt}=a_i(b_i v_i-u_i)$$

Reset rule:

$$v_i\ge 30 \Rightarrow v_i\leftarrow c_i,\quad u_i\leftarrow u_i+d_i$$

Terms: $v_i$ is the reduced voltage-like state, $u_i$ is recovery, $a,b,c,d$ set class-specific dynamics, and $I_i(t)$ combines base drive, recurrent input, and stochastic drive. Worded equation: voltage changes according to intrinsic recovery, a nonlinear voltage term, and input current; threshold events reset the emitter state. Implementation: `jaxfne.emitters.IzhikevichEmitter`, `jaxfne.emitters.simulate_eig_izhikevich`, and Suite No. 2 config builders.

### Source/readout projection

Formal equation:

$$Y_c(t)=\sum_n W_{cn}S_n(t)$$

Terms: $S_n(t)$ is a source proxy from emitter $n$, $W_{cn}$ is a declared projection weight, and $Y_c(t)$ is a channel/contact readout. Worded equation: each readout channel is a weighted sum of source activity. Implementation: `jaxfne.fields.LinearReadout`, `jaxfne.fields.project_laminar_sources`, and `jaxfne.vis.*` figure functions.

## 4. Setup and canonical import

In [ ]:
# Colab-compatible install cell
# !pip install jaxfne

import json
import jaxfne as jtfne


## 5. Configuration constants

In [ ]:
SEED = 7
DURATION_MS = 1000.0
DT_MS = 0.1
DRIVES = {"E": 4.0, "PV": 2.0, "SST": 2.2, "VIP": 2.0}


## 6. One Izhikevich neuron

In [ ]:
cfg_one = jtfne.suite2_single_neuron_config(seed=SEED, duration_ms=DURATION_MS, dt_ms=DT_MS)
model_one = jtfne.construct(cfg_one)
bundle_one = jtfne.suite2_run_bundle(model_one, seed=SEED, duration_ms=DURATION_MS, dt_ms=DT_MS)
bundle_one["signals"].summary()


## 7. Four E/PV/SST/VIP reduced emitters

In [ ]:
cfg_four = jtfne.suite2_four_celltype_config(seed=SEED, duration_ms=DURATION_MS, dt_ms=DT_MS)
model_four = jtfne.construct(cfg_four)
bundle_four = jtfne.suite2_run_bundle(model_four, seed=SEED, duration_ms=DURATION_MS, dt_ms=DT_MS)
model_four.neuron_table()


## 8. net1: 100-emitter uniformly sampled 3D column

In [ ]:
cfg_net1 = jtfne.suite2_net1_config(seed=SEED, n=100, duration_ms=DURATION_MS, dt_ms=DT_MS)
model_net1 = jtfne.construct(cfg_net1)
bundle_net1 = jtfne.suite2_run_bundle(model_net1, seed=SEED, duration_ms=DURATION_MS, dt_ms=DT_MS)
bundle_net1["signals"].summary()


## 9. Drive sweep: E vs PV vs SST vs VIP in net1

In [ ]:
cfg_drive = jtfne.suite2_net1_config(seed=SEED, n=100, duration_ms=DURATION_MS, dt_ms=DT_MS, drives=DRIVES)
model_drive = jtfne.construct(cfg_drive)
bundle_drive = jtfne.suite2_run_bundle(model_drive, seed=SEED, duration_ms=DURATION_MS, dt_ms=DT_MS)
bundle_drive["manifest"]["field_solver_status"]


## 10. Core figures: raster, LFP-like, CSD-like, EEG-proxy, MEG-proxy, EMM-proxy

In [ ]:
fig_raster = jtfne.vis.raster(bundle_drive["signals"], figsize=(8, 4))
fig_lfp = jtfne.vis.lfp_traces(bundle_drive["signals"], figsize=(8, 4))
fig_csd = jtfne.vis.csd_traces(bundle_drive["signals"], figsize=(8, 4))
fig_panel = jtfne.vis.spectrolaminar_suite(bundle_drive["signals"], max_freq_hz=80.0)


## 11. V1-V4: two 400-emitter six-layer columns

The layer fractions below are package defaults for V1. V4 uses a small variant supplied by `suite2_v1_v4_config`.

In [ ]:
FRACS_LAYER = {
    "L1": {"E": .75, "PV": .00, "SST": .00, "VIP": .25},
    "L2": {"E": .75, "PV": .05, "SST": .05, "VIP": .15},
    "L3": {"E": .75, "PV": .10, "SST": .10, "VIP": .05},
    "L4": {"E": .25, "PV": .45, "SST": .15, "VIP": .15},
    "L5": {"E": .15, "PV": .25, "SST": .30, "VIP": .30},
    "L6": {"E": .10, "PV": .20, "SST": .20, "VIP": .50},
}


In [ ]:
cfg_v1v4 = jtfne.suite2_v1_v4_config(seed=SEED, n_per_area=400, duration_ms=DURATION_MS, dt_ms=DT_MS)
model_v1v4 = jtfne.construct(cfg_v1v4)
model_v1v4.summary()


## 12. V1-V4 feedforward and feedback metadata

Pathway grammar: V1 L2/3 excitatory emitters feed V4 L4 excitatory emitters; V4 L2/3 excitatory emitters feed V1 L5/L6 and L1 excitatory emitters.

In [ ]:
cfg_v1v4.metadata["connectivity"]


## 13. Noise tuning with Suite No. 2 AGSDR-Adam utility

In [ ]:
tune_result = jtfne.suite2_tune_noise_agsdr_adam(
    model_drive,
    simulation=jtfne.suite2_simulation(seed=SEED, duration_ms=DURATION_MS, dt_ms=DT_MS),
    target_rate_hz=(5.0, 10.0),
)
tune_result.summary


## 14. Spectrolaminar readout panel

In [ ]:
bundle_v1v4 = jtfne.suite2_run_bundle(model_v1v4, seed=SEED, duration_ms=DURATION_MS, dt_ms=DT_MS)
fig_v1v4 = jtfne.vis.spectrolaminar_suite(bundle_v1v4["signals"], max_freq_hz=80.0)
bundle_v1v4["signals"].summary()


## 15. Manifest and validation summary

In [ ]:
manifest = bundle_v1v4["manifest"]
summary = {
    "n_units": model_v1v4.summary()["n_units"],
    "duration_ms": DURATION_MS,
    "dt_ms": DT_MS,
    "field_solver_status": manifest["field_solver_status"],
}
json.dumps(summary, allow_nan=False, indent=2)


## 16. Interpretation

The upgraded Suite No. 2 demonstrates one-emitter, four-emitter, net1, and V1-V4 workflows through the same public package grammar. The readout panels are reusable across scale because figures consume the `Signals` object and declared metadata rather than notebook-local arrays.

## 17. Failure modes and exercises

Failure modes: unstable input drive, high synchrony, insufficient duration for low-frequency spectra, and oversized dense recurrent matrices on small CPUs.

Exercises: lower PV drive, increase SST drive, compare `net1` before and after noise tuning, vary V4 layer fractions, and repeat the V1-V4 readout with a smaller smoke model before increasing to 400 emitters per area.

## 18. Coverage boundary

This tutorial covers reduced emitters, declared source projection, relative proxy readouts, and package-level figure generation. Solver-tuned amplitudes, subject-specific head geometry, and empirical parameter fitting belong to later workflows.